# Case File: The Comparisons Massacre

So. You opened another one of these.

Run the cell block below and you'll get a nice confident list of "significant" features, practically taking a bow for how many secrets it dug out of a tumor dataset. Twenty nine winners out of forty candidates tested. Feels productive.

Except scroll through that list and you'll find some genuine medical breakthroughs in there. Turns out the patient's ID number has opinions about cancer. Not the tumor's shape, not its texture, not anything a radiologist would recognize. The ID number. The thing a hospital records clerk typed into a spreadadsheet field in 2003.

That is not signal. That is garbage that got a peer reviewer's badge and started introducing itself as a biomarker.

Somewhere in this script the code clearly knows better. There's a whole step that computes a properly strict, properly conservative threshold for exactly this situation. It just doesn't seem to be in charge of anything by the time the final list gets printed.

## Run It

Open a terminal in this folder and run `python comparisons_massacre.py`, or just execute the notebook cells top to bottom. Either way, watch the printed feature list at the end. Count how many of the "significant" biomarkers have the word `id` in their name. That number should be zero. It will not be zero.

Here is the updated text with properly formatted LaTeX equations for clean, clean rendering in Jupyter Notebook markdown cells:

---

## The Math Clue

Sit down. This one's actually important and I'm not going to let you skip it by scrolling past in search of code.

You ran a t-test. One t-test, comparing the malignant group's mean on some feature against the benign group's mean. At a significance threshold of $\alpha = 0.05$, here's what that threshold actually means: **if the feature has genuinely no relationship to diagnosis**, there is a 5% chance, by pure sampling noise, that you'll still see a difference big enough to call "significant." That's not a flaw in the test. That's the definition of $\alpha$. You are explicitly agreeing to be wrong 1 time in 20, in exchange for a fast, cheap decision rule.

One test, 5% false positive rate, fine. Acceptable cost of doing business.

Now do it forty times.

If you run $m$ independent tests, each at significance level $\alpha$, and every single one of those features is genuinely unrelated to the outcome (the null hypothesis is true for all of them), the probability that **at least one** of them shows up "significant" purely by chance is:

$$\text{FWER} = 1 - (1 - \alpha)^m$$

Plug in $\alpha = 0.05$ and $m = 40$:

$$\text{FWER} = 1 - (0.95)^{40} \approx 1 - 0.1285 \approx 0.8715$$

Read that number again. With forty tests at the "standard" 5% threshold, you have roughly an 87% chance of at least one fluke false positive, even in a universe where nothing you're testing has any real connection to the target. You didn't discover a biomarker. You bought a lottery ticket forty times and someone's shocked a ticket won.

This is the family-wise error rate ($\text{FWER}$), the probability of making at least one Type I error across a whole family of tests. It compounds because each additional test is another independent roll of the dice, and the "nothing was actually true" scenario gets more and more likely to produce a false alarm somewhere in the pile the more piles you make.

I've said a lot. THere's a correction for it. It sounds italian. That's all you'll be getting lol. 

## Your Mission

Somewhere in the pipeline below, the correction gets computed, printed, looking entirely respectable, and then quietly ignored by whatever line actually builds the final list. Find where the decision and the calculation stopped talking to each other. I'm not going to underline it for you. Go earn it.

## Dataset

Breast Cancer Wisconsin (Diagnostic) dataset. Download it from Kaggle and place the CSV inside `./dataset/` in this project (the script expects `dataset/data.csv`). You're on your own for the download, this isn't that kind of tutorial.

## Submission

1. Fork this repository.
2. Fix the bug in your fork. One bug, one fix, resist the urge to "clean up" anything else.
3. Open a PR back to this case's folder with a one paragraph explanation of what was wrong and why your fix addresses it.
4. A human reviewer confirms the fix actually resolves the symptom described above.
5. On confirmation, you get the title below. Yes, it's silly. Yes, you'll still want it.

---

### The notebook itself begins below. Try to keep up.

### Step 1: Imports and setup

Nothing to see here. Scipy for the tests, sklearn for the part where we pretend this is a real model. If something's wrong later, it's not going to be in this cell, so don't waste your suspicion on the import statements like every beginner's first instinct tells them to.

In [1]:
# Step 1: imports and setup
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

ALPHA = 0.05
DATA_PATH = "dataset/data.csv"
RANDOM_STATE = 42


### Step 2: Load and clean the data

Boring by design. Load the CSV, map M and B to numbers a computer can use, confirm nothing's missing. If your cleaning step is exciting, you did something wrong somewhere else.

In [2]:
# Step 2: load and clean the raw dataset
def load_data(path):
    """Load the raw Kaggle CSV into a DataFrame."""
    df = pd.read_csv(path)
    # Kaggle export sometimes leaves a trailing unnamed index column
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    return df


def clean_data(df):
    """Encode the target and confirm there are no missing values in the feature columns."""
    df = df.copy()
    df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
    missing = df.isnull().sum().sum()
    if missing > 0:
        raise ValueError(f"Unexpected missing values in dataset: {missing}")
    return df


raw_df = load_data(DATA_PATH)
df = clean_data(raw_df)
print(f"Loaded {len(df)} records, {df['diagnosis'].sum()} malignant, {(df['diagnosis']==0).sum()} benign.")


Loaded 569 records, 212 malignant, 357 benign.


### Step 3: Build the candidate feature pool

Thirty real cell-measurements plus ten features derived from the sample ID, tossed in as a batch-effect check, a real and common thing to test for in medical data. Forty candidates total. Keep that number in your head. It's going to matter a lot more than it looks like it does right now.

In [3]:
# Step 3: assemble the candidate feature pool
def get_measurement_features(df):
    """Return the 30 real-valued cell nucleus measurements."""
    return [c for c in df.columns if c not in ("id", "diagnosis")]


def add_batch_indicator_features(df):
    """
    Derive a handful of features from the sample ID as a standard batch/collection-order
    sanity check. These should carry no biological signal and exist purely to confirm
    the significance screen isn't rubber-stamping noise.
    """
    df = df.copy()
    df["id_mod7"] = df["id"] % 7
    df["id_mod11"] = df["id"] % 11
    df["id_mod13"] = df["id"] % 13
    df["id_mod17"] = df["id"] % 17
    df["id_mod19"] = df["id"] % 19
    df["id_mod23"] = df["id"] % 23
    df["id_mod25"] = df["id"] % 25
    df["id_mod31"] = df["id"] % 31
    df["id_last_digit"] = df["id"] % 10
    df["id_digit_sum"] = df["id"].astype(str).apply(lambda s: sum(int(ch) for ch in s if ch.isdigit()))
    batch_features = ["id_mod7", "id_mod11", "id_mod13", "id_mod17", "id_mod19",
                       "id_mod23", "id_mod25", "id_mod31", "id_last_digit", "id_digit_sum"]
    return df, batch_features


measurement_features = get_measurement_features(df)
df, batch_features = add_batch_indicator_features(df)
candidate_features = measurement_features + batch_features
print(f"Testing {len(candidate_features)} candidate features "
      f"({len(measurement_features)} measurements + {len(batch_features)} batch indicators).")


Testing 40 candidate features (30 measurements + 10 batch indicators).


### Step 4: Test each feature individually

One t-test per candidate, malignant group against benign group. Forty of them. You already know what forty independent tests at the standard threshold gets you, because you just read several hundred words about it and I know you didn't skim, right, right.

In [4]:
# Step 4: run an independent-samples t-test for every candidate feature
def run_univariate_tests(df, feature_cols, target_col="diagnosis"):
    """Compare the malignant and benign groups on each candidate feature."""
    malignant = df[df[target_col] == 1]
    benign = df[df[target_col] == 0]
    rows = []
    for feature in feature_cols:
        t_stat, p_val = stats.ttest_ind(malignant[feature], benign[feature], equal_var=False)
        rows.append({"feature": feature, "t_stat": t_stat, "p_value": p_val})
    return pd.DataFrame(rows)


results_df = run_univariate_tests(df, candidate_features)


### Step 5: Apply the correction

Here it is. The corrected threshold, computed properly, printed proudly. Everything upstream of this cell is doing exactly what it should. Enjoy this feeling of a well-behaved pipeline, it's the last calm moment you're going to have for a while.

In [5]:
# Step 5: correct for the number of hypotheses tested
def apply_bonferroni_correction(results_df, alpha=ALPHA):
    """Attach a Bonferroni-corrected threshold and flag which features clear each bar."""
    results_df = results_df.copy()
    n_tests = len(results_df)
    corrected_alpha = alpha / n_tests
    results_df["corrected_alpha"] = corrected_alpha
    results_df["significant_raw"] = results_df["p_value"] < alpha
    results_df["significant_corrected"] = results_df["p_value"] < corrected_alpha
    return results_df, corrected_alpha, n_tests


results_df, corrected_alpha, n_tests = apply_bonferroni_correction(results_df)
print(f"Ran {n_tests} tests. Corrected significance threshold: {corrected_alpha:.6f}")


Ran 40 tests. Corrected significance threshold: 0.001250


### Step 6: Pick the winners

The list that actually matters. Whatever comes out of this function is what the rest of the pipeline treats as real. No pressure.

In [6]:
# Step 6: extract the feature names that passed the significance screen
def select_features(results_df):
    """Return the list of feature names flagged as significant."""
    selected = results_df[results_df["significant_corrected"]]["feature"].tolist()
    return selected


selected_features = select_features(results_df)
print(f"Selected {len(selected_features)} of {n_tests} candidate features.")
print(selected_features)


Selected 29 of 40 candidate features.
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'radius_se', 'perimeter_se', 'area_se', 'compactness_se', 'concavity_se', 'concave points_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst', 'id_mod25', 'id_mod31', 'id_digit_sum']


### Step 7: Train something and look at the damage

A quick logistic regression, because the point of finding "significant" features is usually to use them for something. Watch the last two printed lines closely. One of them should be an empty list. It is not going to be an empty list.

In [7]:
# Step 7: train a quick classifier on the surviving features
def train_and_evaluate(df, feature_cols, target_col="diagnosis"):
    """Fit a simple logistic regression on the selected features and report holdout accuracy."""
    X = df[feature_cols]
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
    )
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(max_iter=5000)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    return accuracy_score(y_test, preds)


accuracy = train_and_evaluate(df, selected_features)
print(f"Holdout accuracy using selected features: {accuracy:.4f}")

flagged_noise = [f for f in selected_features if f.startswith("id_")]
print(f"Batch-indicator features that made the cut: {flagged_noise}")


Holdout accuracy using selected features: 0.9720
Batch-indicator features that made the cut: ['id_mod25', 'id_mod31', 'id_digit_sum']


<details>
<summary><b>Click to reveal your reward (only open this after your fix is confirmed, cheater)</b></summary>

## MOCK CEREMONY: A Title Is Bestowed

By the somewhat dubious authority vested in me as Head TA of a repository that exists specifically to lie to you, I hereby promote you to:

### **Grand Inquisitor of the Family-Wise Error Rate, Bane of Spurious Correlations, Sworn Enemy of the Sample ID Column**

You found it. You found the line where a perfectly good correction got computed, printed, admired, and then completely ignored by the one function whose entire job was to use it. Somewhere out there, a patient ID column is filing an appeal, claiming it really did feel like it had something to say about oncology. It did not. You knew it did not. That's the job.

Go forth and distrust every p-value you meet at a party.

</details>